In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.chdir('/zhome/71/c/146676/main/')
import SimpleITK as sitk
import loader_XA_to_NA
import importlib
importlib.reload(loader_XA_to_NA)

In [ ]:
reg = loader_XA_to_NA.load_registered_data(compression = 1)

In [ ]:
plt.imshow(reg.as_array(reg.fixed)[700])
plt.clim([-0.02,0.1])
plt.show()

plt.imshow(reg.as_array(reg.moving)[700])
plt.clim([-0.02,0.1])
plt.show()

plt.imshow(reg.as_array(reg.fixed)[:,200])
plt.clim([-0.02,0.1])
plt.show()

plt.imshow(reg.as_array(reg.moving)[:,200])
plt.clim([-0.02,0.1])
plt.show()

In [ ]:
plt.imshow((reg.as_array(reg.fixed)[:,150]<0.02)*1.0 - (reg.as_array(reg.moving)[:,150]<0.02)*1.0)

In [117]:
threshold = [0.02, 0.02]
factor = 4
reg.compute_stone_boundaries(thresholds = threshold, factor = factor) # saved in reg.fixed_seg and reg.moving_seg
initial_mask = reg.fixed_seg
mask = sitk.Resample(
    initial_mask,
    reg.fixed,
    sitk.Transform(),
    sitk.sitkNearestNeighbor,  # Use nearest neighbor to preserve binary mask values
    initial_mask.GetPixelID()
)

mask = reg.as_array(mask)
fixed = reg.as_array(reg.fixed)
moving = reg.as_array(reg.moving)

In [ ]:
sampling_rate = 1
shape = np.shape(mask)
random_mask = np.random.rand(*shape) < sampling_rate
mask = mask*random_mask*(fixed>0.0001)*np.logical_or(moving < -0.00001, moving > 0.00001)
mask = mask.astype(bool)


fixed_values = fixed[mask]
moving_values = moving[mask]

prefix = '/dtu-compute/msaca/output/'

# Create 2d histogram/heatmap of non postfiltered images
path_image_log = prefix + 'heatmap_log_01.png'
path_image = prefix + 'heatmap_01.png'
path_matrix = prefix + 'heatmap_matrix_01.npy'
title = 'FBP recons, slice A, background masked, no postfilter'

In [119]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

def add_squares(x, y):
  
    """
    Add squares to an existing figure based on specified intervals, with automatic color assignment.
    
    Parameters:
    - x: List of intervals for the x-axis, e.g., [[x1_s, x1_e], [x2_s, x2_e], ...].
    - y: List of intervals for the y-axis, e.g., [[y1_s, y1_e], [y2_s, y2_e], ...].
    - cmap: Matplotlib colormap for generating colors (default: "tab10").
    - linewidth: Thickness of the square outlines (default: 2).

    Returns:
    - colors: List of colors assigned to each square.
    """
    cmap = "tab10"
    if len(x) != len(y):
        raise ValueError("The number of x-intervals and y-intervals must be the same.")
    
    ax = plt.gca()  # Get the current axes
    num_squares = len(x)
    colors = cm.get_cmap(cmap, num_squares).colors  # Generate distinct colors from the colormap
    
    for i, (x_interval, y_interval) in enumerate(zip(x, y)):
        # Ensure intervals are valid
        if len(x_interval) != 2 or len(y_interval) != 2:
            raise ValueError("Each interval must have exactly two elements: [start, end].")
        
        # Add a rectangle for each interval pair
        rectangle = plt.Rectangle(
            (x_interval[0], y_interval[0]),  # Bottom-left corner
            x_interval[1] - x_interval[0],  # Width
            y_interval[1] - y_interval[0],  # Height
            edgecolor=colors[i],
            facecolor="none",
            linewidth=2,
        )
        ax.add_patch(rectangle)

    colormaps = []
    for i, color in enumerate(colors):
        # Define the color map
        name_prefix = "custom"
        factor = 1.5
        cmap = mcolors.LinearSegmentedColormap.from_list(
            f"{name_prefix}_{i}",
            [(0, 0, 0), color]  # Transition from black to the given color
        )
        cdict = cmap._segmentdata
        # Modify the colors (this is done for each color channel: red, green, blue)
        for channel in ['red', 'green', 'blue']:
            # Scale each color in the channel towards 1 (white)
            for i, (t, c1, c2) in enumerate(cdict[channel]):
                new_c1 = min(c1 * factor, 1.0)
                new_c2 = min(c2 * factor, 1.0)
                cdict[channel][i] = (t, new_c1, new_c2)
        b_cmap = mcolors.LinearSegmentedColormap(cmap.name + '_brightened', segmentdata=cdict)
        colormaps.append(b_cmap)
    return colormaps


In [110]:
def heatmap(fixed_values, moving_values,neutron=None, xray = None):
    bins = 400
    nticks = 15
    range_fixed = [-0.01, 0.11]
    range_moving = [-0.11, 0.11]

    # Create the heatmap
    heatmap, xedges, yedges = np.histogram2d(
        fixed_values, moving_values, bins=(bins, bins),
        range=[range_fixed, range_moving]
    )
    log_heatmap = np.log1p(heatmap)

    # Plot the heatmap
    plt.figure(figsize=(8, 8))
    plt.imshow(
        log_heatmap.T, origin="lower", aspect="auto", cmap="YlGnBu",
        extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]]  # Set extent to match data range
    )
    plt.colorbar(label="log-Frequency")
    plt.xlabel("Fixed: Neutron")
    plt.ylabel("Moving: Xray")
    plt.title("Heatmap with Highlighted Square")

    # Set custom ticks
    x_tick_positions = np.linspace(xedges[0], xedges[-1], num=nticks)
    y_tick_positions = np.linspace(yedges[0], yedges[-1], num=nticks)
    x_tick_labels = [f"{x:.3f}" for x in x_tick_positions]
    y_tick_labels = [f"{y:.3f}" for y in y_tick_positions]
    plt.xticks(x_tick_positions, x_tick_labels)
    plt.yticks(y_tick_positions, y_tick_labels)

    if neutron is not None:
        colors = add_squares(x=neutron, y=xray)
    # Show the plot
    plt.show()
    if neutron is not None:
        return colors


In [ ]:
neutron = [[0, 0.04], [0,0.07], [0.05, 0.1], [0.01, 0.06]]
xray = [[0.01,0.025], [-0.05, 0], [-0.09, 0.09], [0.05, 0.1]]
colors = heatmap(fixed_values, moving_values,neutron=neutron, xray = xray)

In [ ]:
def display_segmentation(neutron_values, xray_values, neutron_intervals, xray_intervals):
    num_plots = len(xray_intervals)  # Adjust this for the number of plots you want
    plots_per_row = 3

    # Calculate the number of rows needed
    num_rows = (num_plots + plots_per_row - 1) // plots_per_row  # Ceiling division

    # Create the figure with subplots
    fig, axes = plt.subplots(num_rows, plots_per_row, figsize=(22.5, 7.5 * num_rows))  # Adjust figure size as needed
    axes = axes.flatten()  # Flatten the axes array for easier indexing
    # Set the figure background to black
    fig.patch.set_facecolor('black')

    # Loop through each plot
    for i in range(num_plots):
        ax = axes[i]
        segm = mask*np.logical_and(neutron_values>neutron_intervals[i][0],neutron_values<neutron_intervals[i][1])*np.logical_and(xray_values>xray_intervals[i][0],xray_values<xray_intervals[i][1])
        proj_segm = np.sum(segm,axis=1)/np.shape(segm)[1]
        ax.imshow(proj_segm, cmap = colors[i])
        # Remove axes and set background color to black
        ax.axis('off')
        ax.set_facecolor('black')  # Set subplot background color

    # Hide any extra subplots if num_plots is not a multiple of plots_per_row
    for j in range(num_plots, len(axes)):
        fig.delaxes(axes[j])


display_segmentation(neutron_values = fixed, xray_values = moving, neutron_intervals = neutron, xray_intervals = xray)

In [ ]:
# Apply gaussian filtering before making the heatmap
fixed_filtered_ = sitk.SmoothingRecursiveGaussian(reg.fixed, sigma=0.002)
moving_filtered_ = sitk.SmoothingRecursiveGaussian(reg.moving, sigma=0.002)
fixed_filtered = reg.as_array(fixed_filtered_)
moving_filtered = reg.as_array(moving_filtered_)
fixed_values = fixed_filtered[mask]
moving_values = moving_filtered[mask]

xray = [[0.016,0.02], [-0.016, 0.016], [0.02, 0.03], [0.03,0.04],[0.02, 0.03], [0.03,0.04] ]
neutron = [[0, 0.03], [0,0.01], [0,0.025], [0,0.025],[0.035,0.065], [0.035,0.065]]
colors = heatmap(fixed_values, moving_values,neutron=neutron, xray = xray)

In [ ]:
display_segmentation(neutron_values = fixed_filtered, xray_values = moving_filtered, neutron_intervals = neutron, xray_intervals = xray)